In [1]:
import digitalhub as dh

project = dh.get_or_create_project("test-tvm")

In [28]:
model = dh.log_model(
    project="test-tvm",
    name="xinet-pose-224",
    source="./XiNet-s-pose-224.onnx",
    framework="onnx"
)

In [4]:
xinet_tvm_func = project.new_function(name="xinet-pose-tvm", kind="tvm",
                                  model="store://test-tvm/model/model/xinet-pose-224", 
                                  format="auto")

In [ ]:
xinet_tvm_build = xinet_tvm_func.run(
    action="build",
    resources={"cpu":"1", "mem":"2Gi"},
    wait=True
)

In [ ]:
xinet_tvm_build = xinet_tvm_func.run(
    action="compile",
    target_architecture="x86",
    tag="x86",
    opt_level=3,
    resources={"cpu":"1", "mem":"2Gi"},
    wait=True
)

In [ ]:
xinet_tvm_func.run(action="serve", resources={"cpu":"1", "mem":"1Gi"}, wait=True)

In [ ]:
import json
import urllib.request

import numpy as np
from PIL import Image
    
URL = "http://s-test-tvm-xinet-pose-tvm-latest.dev-platform:8080"   # port-forward, or http://localhost:8080 with Docker
MODEL = "xinet-pose-tvm"          # the served name

# 1. Read the model signature
meta = json.load(urllib.request.urlopen(f"{URL}/v2/models/{MODEL}"))
spec = meta["inputs"][0]
_, _, height, width = spec["shape"]          # [1, 3, 640, 640]

# 2. Prepare the picture: resize, scale to [0, 1], HWC -> NCHW
image = Image.open("bus.jpg").convert("RGB").resize((width, height))
tensor = (np.asarray(image, dtype=np.float32) / 255.0).transpose(2, 0, 1)[None]

# 3. Call the Open Inference v2 endpoint
body = {"inputs": [{"name": spec["name"], "datatype": "FP32",
                    "shape": list(tensor.shape), "data": tensor.ravel().tolist()}]}
request = urllib.request.Request(f"{URL}/v2/models/{MODEL}/infer", data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
result = json.load(urllib.request.urlopen(request, timeout=300))
print(f"{len(result["outputs"][0]["data"])}")


In [10]:
func = project.new_function(name="xinet-pose",
                            kind="openinference",
                            python_version="PYTHON3_13",
                            code_src="xinet",
                            handler="xinet_tvm_handler:handler",
                            init_function="init_context",
                            model_name="XiNet-s-pose-224",
                            inputs=[{
                                "name": "image",
                                "datatype": "UINT8",
                                "shape": [-1,-1]
                            }],
                            outputs=[{
                                "name": "image",
                                "datatype": "UINT8",
                                "shape": [-1,-1]
                            }],
                            requirements=["opencv-python==4.13.0.92", "numpy==2.4.3", "onnxruntime==1.24.3", "matplotlib==3.10.8"]
                           )

In [ ]:
build = func.run(
    "build",
    instructions=[
        "apt-get update && apt-get install -y libgl1 libglib2.0-0 libsm6 libxext6 libxrender1 libfontconfig1 libice6"
    ],
    wait=True
)

In [13]:
run = func.run(action="serve", resources={"mem": "1Gi", "disk": "1Gi"}, init_parameters={"tvm_func":"http://s-test-tvm-xinet-pose-tvm-latest.dev-platform:8080/v2/models/xinet-pose-tvm"})

In [15]:
stream_func = project.new_function("videostream", kind="container", image="alexxit/go2rtc", command="/bin/bash", code_src="go2rtc")

In [ ]:
stream_func.run(action="serve", 
                service_ports=[{"port": 1984, "target_port": 1984}], 
                fs_group=8877, run_as_user=8877, run_as_group=8877, 
                args=["/shared/lunch_go2rtc.sh"],
                wait=True)

In [14]:
graph_func = project.new_function("xinet-pipeline", kind="servicegraph", code_src="pipeline.yaml")

In [17]:
graph_run = graph_func.run(action="serve", parameters={
    "input.url": "http://s-test-tvm-videostream-latest.dev-platform:1984/api/stream.mjpeg?src=example",
    "xinet-pose-service.address": "s-test-tvm-xinet-pose-latest.dev-platform:9000"
}, service_ports=[{"port": 7777, "target_port": 7777}])

In [ ]:
# dhcli login
# dhcli port-forward -p test-tvm -f xinet-pipeline -l 7777
# http://localhost:7777/stream